# Day 3 — Token-level language identification
1. Char n-gram + Logistic Regression baseline (CPU).
2. Fine-tune `FacebookAI/xlm-roberta-base` (GPU): lr 2e-5, batch 4 × grad-accum 4, max length 128, 3 epochs, warmup 0.1, wd 0.01, FP16, selection by validation macro-F1.

Label alignment: word label on the **first subword**; continuation subwords and special tokens = -100.

In [ ]:
# --- Setup (run once per Colab session) ---
REPO_URL = "https://github.com/arkalatua/Indic_Switch.git"
import os, sys, subprocess
if "google.colab" in sys.modules:
    if not os.path.exists("/content/Indic_Switch"):
        subprocess.run(["git", "clone", REPO_URL, "/content/Indic_Switch"], check=True)
    %cd /content/Indic_Switch
    !pip install -q -r requirements.txt
    # Optional: persist models/results across sessions
    # from google.colab import drive; drive.mount("/content/drive")
else:
    os.chdir(os.path.abspath(".."))  # running locally from notebooks/
sys.path.insert(0, os.getcwd())

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

In [ ]:
import os
if not os.path.exists("data/processed/lid_train.jsonl"):
    !python scripts/prepare_comi_lingua.py

## Baselines

In [ ]:
!python scripts/train_ngram_lid.py
!python scripts/evaluate_lid.py --model models/ngram_lid.pkl --split test
!python scripts/evaluate_lid.py --model rule --split test

## Check label alignment on one example

In [ ]:
from transformers import AutoTokenizer
from src.language_identification import align_labels, LABEL2ID, read_lid_jsonl
tok = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-base")
ex = read_lid_jsonl("data/processed/lid_train.jsonl")[0]
enc = tok(ex["tokens"], is_split_into_words=True)
labels = align_labels(enc.word_ids(), [LABEL2ID[t] for t in ex["tags"]])
for t, w, l in zip(enc.tokens(), enc.word_ids(), labels): print(f"{t:15s} word={w}  label={l}")

## Fine-tune XLM-R (OOM? use `--max-length 96`)

In [ ]:
!python scripts/train_xlmr_lid.py --batch-size 4 --grad-accum 4 --max-length 128 --epochs 3

In [ ]:
!python scripts/evaluate_lid.py --model models/lid_model --split test

## Compare and inspect errors

In [ ]:
import json, pandas as pd
rows = []
for name in ["script_rule", "char_ngram_lr", "xlmr"]:
    try: r = json.load(open(f"experiments/lid_{name}_test.json"))
    except FileNotFoundError: continue
    rows.append({"model": name, "macro_F1": r["lid"]["macro_f1"], "acc": r["lid"]["accuracy"],
                 **{f"F1_{k}": v["f1"] for k, v in r["lid"]["per_class"].items()},
                 "boundary_F1": r["boundary"]["f1"], "tok/s": r["efficiency"]["tokens_per_second"]})
display(pd.DataFrame(rows).round(4))
r = json.load(open("experiments/lid_xlmr_test.json"))
display(pd.DataFrame(r["top_confusions"][:20])); display(pd.DataFrame(r["error_examples"][:20]))

In [ ]:
# Save the checkpoint outside Git (e.g. Google Drive) — it is ~1 GB
# !cp -r models/lid_model /content/drive/MyDrive/indicswitch_lid_model